## 🔧 0. Preparación del Entorno e Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Configuración de salida y reproducibilidad
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
SEED = 42
np.random.seed(SEED)

print("✅ Entorno de Preprocesamiento listo.")

✅ Entorno de Preprocesamiento listo.


In [3]:
# 1. Definición de rutas
RAW_MASTER_PATH = r"C:\Users\Carlos\Documents\Curso_Analisis_Data_bootcamp_Upgrade_Hub\Inteligencia_Producto_E_Commerce\data\processed\ecommerce_master_tablon.csv"

if not os.path.exists(RAW_MASTER_PATH):
    raise FileNotFoundError(f"No se encontró el tablón maestro en: {RAW_MASTER_PATH}")

# 2. Carga del dataset
df = pd.read_csv(RAW_MASTER_PATH)
print(f"📊 Dataset cargado correctamente: {df.shape[0]} filas y {df.shape[1]} columnas.")

📊 Dataset cargado correctamente: 100000 filas y 31 columnas.


## 🛠️ Transformación y limpieza

In [13]:
# =====================================================================
# FASE INITIAL: SANEAR, ESTANDARIZAR Y DEPURAR EL DATASET MAESTRO
# =====================================================================

def higienizar_dataset_maestro(df_original):
    """
    Trabaja sobre una copia para proteger el origen. Estandariza nombres de 
    columnas (minúsculas, sin espacios, sin caracteres especiales) y elimina duplicados.
    """
    print("="*60)
    print("🧼 INICIANDO SANEAMIENTO Y LIMPIEZA DE INFRAESTRUCTURA")
    print("="*60)
    
    # 1. Proteger el DataFrame original trabajando sobre una copia limpia
    df_trabajo = df_original.copy()
    filas_iniciales, columnas_iniciales = df_trabajo.shape
    print(f"📦 Dataset de trabajo original: {filas_iniciales:,} filas × {columnas_iniciales} columnas")
    
    # 2. ESTANDARIZAR NOMBRES DE COLUMNAS 
    columnas_viejas = df_trabajo.columns.tolist()
    
    df_trabajo.columns = (
        df_trabajo.columns
        .str.strip()                             # Eliminar espacios en blanco en los extremos
        .str.lower()                             # Convertir todo a minúsculas
        .str.replace(' ', '_')                   # Reemplazar espacios intermedios por guiones bajos
        .str.replace(r'[^\w]', '', regex=True)  # Eliminar caracteres especiales o acentos conflictivos
    )
    
    print("\n🔄 Estandarización de columnas completada.")
    print(f"   ├─ Muestra Antes : {columnas_viejas[:3]}")
    print(f"   └─ Muestra Ahora : {df_trabajo.columns.tolist()[:3]}")
    
    # 3. ELIMINAR FILAS DUPLICADAS CRUCIALES
    df_trabajo_limpio = df_trabajo.drop_duplicates()
    filas_finales = len(df_trabajo_limpio)
    duplicados_detectados = filas_iniciales - filas_finales
    porcentaje_duplicados = (duplicados_detectados / filas_iniciales) * 100
    
    print(f"\n🔁 Purga de registros duplicados:")
    print(f"   ├─ Filas eliminadas : {duplicados_detectados:,} (({porcentaje_duplicados:.2f}%))")
    print(f"   └─ Filas remanentes : {filas_finales:,}")
    print("="*60)
    
    return df_trabajo_limpio

# Ejecutamos la higienización y creamos nuestro DataFrame oficial de trabajo
df_saneado = higienizar_dataset_maestro(df)

🧼 INICIANDO SANEAMIENTO Y LIMPIEZA DE INFRAESTRUCTURA
📦 Dataset de trabajo original: 100,000 filas × 31 columnas

🔄 Estandarización de columnas completada.
   ├─ Muestra Antes : ['interaction_id', 'user_id', 'product_id']
   └─ Muestra Ahora : ['interaction_id', 'user_id', 'product_id']

🔁 Purga de registros duplicados:
   ├─ Filas eliminadas : 0 ((0.00%))
   └─ Filas remanentes : 100,000


## 🛠️ Auditoría de Calidad de Datos (Data Quality Framework):

Nota importnte: nunca se empieza a transformar un dataset unificado sin antes realizar una auditoría de calidad de datos de entrada (Data Quality Audit).

In [14]:
# =====================================================================
# DETECCIÓN DE CALIDAD AVANZADA POST-SANEAMIENTO
# =====================================================================

def auditoria_calidad_datos_avanzada(df_evaluación):
    """
    Diagnóstico integral de calidad. Evalúa nulos, tipos de datos,
    métricas de dispersión, outliers por IQR y duplicados globales.
    """
    # 1. Cálculos de duplicados a nivel general
    total_filas = len(df_evaluación)
    total_duplicados = df_evaluación.duplicated().sum()
    
    print("="*75)
    print("🔬 REPORTE AVANZADO DE CALIDAD DE DATOS (DATA QUALITY AUDIT)")
    print("="*75)
    print(f"📈 Dimensiones actuales : {total_filas:,} filas | {df_evaluación.shape[1]} columnas")
    print(f"🔁 Duplicados globales  : {total_duplicados:,} filas detectadas en el set de evaluación")
    print("="*75)
    
    reporte_columnas = []
    
    for col in df_evaluación.columns:
        valores_nulos = df_evaluación[col].isna().sum()
        porcentaje_nulos = (valores_nulos / total_filas) * 100
        valores_unicos = df_evaluación[col].nunique()
        tipo_dato = df_evaluación[col].dtype
        
        media, mediana, minimo, maximo, q25, q75, flag_outliers = "N/A", "N/A", "N/A", "N/A", "N/A", "N/A", "N/A"
        
        if np.issubdtype(tipo_dato, np.number):
            media = df_evaluación[col].mean()
            mediana = df_evaluación[col].median()
            minimo = df_evaluación[col].min()
            maximo = df_evaluación[col].max()
            q25 = df_evaluación[col].quantile(0.25)
            q75 = df_evaluación[col].quantile(0.75)
            
            # Criterio estadístico IQR para Outliers
            iqr = q75 - q25
            limite_inferior = q25 - 1.5 * iqr
            limite_superior = q75 + 1.5 * iqr
            
            outliers_count = df_evaluación[(df_evaluación[col] < limite_inferior) | (df_evaluación[col] > limite_superior)][col].count()
            flag_outliers = f"Sí ({outliers_count:,})" if outliers_count > 0 else "No"
        
        ejemplo_valores = df_evaluación[col].dropna().unique()[:3].tolist()
        
        reporte_columnas.append({
            'Columna': col,
            'Tipo': str(tipo_dato),
            'Nulos (Cant.)': valores_nulos,
            'Nulos (%)': porcentaje_nulos,
            'Valores Únicos': valores_unicos,
            'Media': media,
            'Mediana': mediana,
            'Mínimo': minimo,
            'Máximo': maximo,
            'Q25': q25,
            'Q75': q75,
            '¿Tiene Outliers?': flag_outliers,
            'Ejemplos': ejemplo_valores
        })
    
    df_reporte = pd.DataFrame(reporte_columnas)
    return df_reporte

# Lanzamos el diagnóstico definitivo sobre el dataframe ya higienizado
reporte_final = auditoria_calidad_datos_avanzada(df_saneado)
reporte_final.sort_values(by='Nulos (%)', ascending=False)

🔬 REPORTE AVANZADO DE CALIDAD DE DATOS (DATA QUALITY AUDIT)
📈 Dimensiones actuales : 100,000 filas | 31 columnas
🔁 Duplicados globales  : 0 filas detectadas en el set de evaluación


,Columna,Tipo,Nulos (Cant.),Nulos (%),Valores Únicos,Media,Mediana,Mínimo,Máximo,Q25,Q75,¿Tiene Outliers?,Ejemplos
26,rating_avg,float64,11615,11.62,21,4.09,4.10,1.00,5.00,3.90,4.30,"Sí (13,536)","[4.2, 2.5, 4.5]"
1,user_id,object,0,0.00,6944,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[8a49a58e-7e31-4616-8ac8-b1472ed4f5d8, 1a79b3a..."
0,interaction_id,object,0,0.00,100000,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[2ece1c7b-3244-4eeb-8d3e-f1b0cc09f20f, bef097f..."
3,session_id,object,0,0.00,19315,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[0003375f-8f69-44e1-9429-ec083de18495, 0003761..."
4,interaction_type,object,0,0.00,6,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[view, add_to_wishlist, remove_from_wishlist]"
5,timestamp,object,0,0.00,96257,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[2025-03-13 08:29:18.000000832, 2025-03-13 08:..."
2,product_id,object,0,0.00,967,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[d3701c21-709a-4090-bab2-020261ce8a92, b1a8533..."
7,user_id_sesion,object,0,0.00,6944,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[8a49a58e-7e31-4616-8ac8-b1472ed4f5d8, 1a79b3a..."
8,start_time,object,0,0.00,19315,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[2025-03-13 08:29:18.000000832, 2026-05-07 16:..."
9,device_type,object,0,0.00,3,N/A,N/A,N/A,N/A,N/A,N/A,N/A,"[mobile, desktop, tablet]"


## 🏷️ PURGA DE METADATOS (REDUCCIÓN DE DIMENSIONALIDAD INICIAL ) Y CLASIFICACIÓN AUTOMATIZADA DE PREDICTORES 

In [15]:
# =====================================================================
# FASE 1: PURGA DE METADATOS Y CLASIFICACIÓN AUTOMATIZADA DE PREDICTORES
# =====================================================================

# 1. Definición defensiva de columnas a descartar (IDs y textos libres)
# 🔍 NOTA: Ahora van en minúsculas porque df_saneado ya las estandarizó.
columnas_a_descartar = [
    'interaction_id', 'user_id', 'product_id', 'session_id', 'user_id_sesion',
    'product_name', 'product_description',
    'timestamp', 'start_time', 'signup_date', 'date_added',
    'dwell_time_ms'
]

# 2. Limpieza de metadatos utilizando el DataFrame SANEADO e HIGIENIZADO
df_filtrado = df_saneado.drop(columns=[col for col in columnas_a_descartar if col in df_saneado.columns])

# 3. Aislamiento del Target (También en minúsculas)
TARGET = 'is_converted'
X = df_filtrado.drop(columns=[TARGET])
y = df_filtrado[TARGET]

# 4. El Split Sagrado (80% Train / 20% Test) con estratificación
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

# 5. DETECCIÓN AUTOMÁTICA Y GENÉRICA POR TIPO TÉCNICO
variables_numericas = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
variables_categoricas = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# 6. REPORTE DE INFRAESTRUCTURA METODOLÓGICA
print("="*60)
print("🔬 REPORT DE ESTRUCTURA METODOLÓGICA (FASE 1)")
print("="*60)
print(f"📈 Matriz de Entrenamiento (X_train): {X_train.shape[0]:,} filas y {X_train.shape[1]} predictores.")
print(f"📈 Matriz de Prueba        (X_test) : {X_test.shape[0]:,} filas y {X_test.shape[1]} predictores.")
print(f"🎯 Variable Objetivo       (Target) : '{TARGET}' ({y_train.dtype})")
print(f"   └─ Balance en Train: {y_train.value_counts(normalize=True).to_dict()}")
print("="*60)

print(f"🔢 VARIABLES NUMÉRICAS DETECTADAS ({len(variables_numericas)}):")
print("-" * 60)
for idx, col in enumerate(variables_numericas, 1):
    print(f" [{idx:02d}] Column: '{col}' -> Tipo técnico: {X_train[col].dtype}")

print("\n🔤 VARIABLES CATEGÓRICAS DETECTADAS ({len(variables_categoricas)}):")
print("-" * 60)
for idx, col in enumerate(variables_categoricas, 1):
    print(f" [{idx:02d}] Column: '{col}' -> Tipo técnico: {X_train[col].dtype}")
print("="*60)

🔬 REPORT DE ESTRUCTURA METODOLÓGICA (FASE 1)
📈 Matriz de Entrenamiento (X_train): 80,000 filas y 18 predictores.
📈 Matriz de Prueba        (X_test) : 20,000 filas y 18 predictores.
🎯 Variable Objetivo       (Target) : 'is_converted' (bool)
   └─ Balance en Train: {False: 0.8905, True: 0.1095}
🔢 VARIABLES NUMÉRICAS DETECTADAS (6):
------------------------------------------------------------
 [01] Column: 'age' -> Tipo técnico: int64
 [02] Column: 'price' -> Tipo técnico: float64
 [03] Column: 'rating_avg' -> Tipo técnico: float64
 [04] Column: 'review_count' -> Tipo técnico: int64
 [05] Column: 'stock_quantity' -> Tipo técnico: int64
 [06] Column: 'dwell_time_secs' -> Tipo técnico: float64

🔤 VARIABLES CATEGÓRICAS DETECTADAS ({len(variables_categoricas)}):
------------------------------------------------------------
 [01] Column: 'interaction_type' -> Tipo técnico: object
 [02] Column: 'device_type' -> Tipo técnico: object
 [03] Column: 'referrer_source' -> Tipo técnico: object
 [04] Co

## 🧑‍💻 Código de la Fase 2: Imputación Inteligente por Categoría

🗺️ ¿En qué etapa del proyecto estamos y hacia dónde vamos?
Estamos en la fase de Preprocesamiento de Datos (o Data Preparation).

- El objetivo de esta etapa: Dejar las matrices de datos (X_train y X_test) impecables, en formato 100% numérico y sin un solo vacío, para que los modelos de Machine Learning puedan procesarlas sin lanzar errores matemáticos.

- Hacia dónde vamos: Una vez que terminemos de limpiar y transformar las variables (en los siguientes pasos escalaremos los números y codificaremos los textos), pasaremos directamente a la Fase de Modelado, donde la Inteligación Artificial leerá estos datos limpios para aprender a predecir si un cliente va a comprar o no (is_converted).

🎯 ¿Cuál es el objeto de este script específico?
El objeto del script es neutralizar los valores nulos (NaN) de la columna rating_avg de forma realista y segura.

Como tu auditoría inicial detectó que más de la mitad de los productos no tenían calificación, teníamos que rellenar esos huecos. El script hace una imputación condicionada por contexto: en lugar de inventar una nota promedio para todo el e-commerce, el código va departamento por departamento.
Si a un artículo de Automotive (Automoción) le falta su calificación, el script mira el comportamiento de su propio grupo y le asigna un 4.70. En cambio, si el producto es un libro (Books), le asigna un 3.90. Esto refleja la realidad comercial del catálogo.

In [19]:
# =====================================================================
# FASE 2: IMPUTACIÓN INTELIGENTE DE NULOS (REVISIÓN DE INGENIERÍA)
# =====================================================================

def ejecutar_imputacion_inteligente(X_train_df, X_test_df, col_agrupadora, col_a_imputar):
    """
    Función empaquetada para rellenar nulos de forma quirúrgica.
    Usa una columna 'guía' (col_agrupadora) para reparar la columna 'dañada' (col_a_imputar).
    """
    # PASO A: Creamos copias exactas en la memoria de Python.
    # Esto se hace para que Pandas no nos llance una advertencia (Warning) de que estamos
    # alterando los datos originales sin permiso.
    X_train_out = X_train_df.copy()
    X_test_out = X_test_df.copy()
    
    # PASO B: Contamos cuántos nulos hay en este preciso instante antes de empezar.
    # Guardamos el número en una variable para el reporte final.
    nulos_train_inicial = X_train_out[col_a_imputar].isna().sum()
    nulos_test_inicial = X_test_out[col_a_imputar].isna().sum()
    
    # PASO C: LA BRÚJULA ESTADÍSTICA (Calculada SOLO con el 80% de Train)
    # .groupby(col_agrupadora) -> Junta los datos por 'category' (crea grupos de Electronics, Books, etc.)
    # [col_a_imputar].median() -> De cada grupo, calcula la MEDIANA de su 'rating_avg'
    # Esto genera un diccionario/mapa mental. Ej: {'Books': 3.90, 'Electronics': 4.10}
    medianas_por_categoria = X_train_out.groupby(col_agrupadora)[col_a_imputar].median()
    
    # Red de seguridad: Calculamos la mediana de TODA la columna por si acaso en el set
    # de Test aparece una categoría nueva que no existía en Train.
    mediana_global_train = X_train_out[col_a_imputar].median()
    
    # PASO D: EL ALGORITMO DE ASIGNACIÓN (La regla que se aplica a cada fila)
    def rellenar_vacio(fila, mapa_medianas, respaldo_global):
        # 1. Si la celda de la columna (rating_avg) NO es nula, no hacemos nada, devolvemos su valor real.
        if not pd.isna(fila[col_a_imputar]):
            return fila[col_a_imputar]
        
        # 2. Si la celda SÍ es nula (NaN), extraemos la categoría de esa fila (ej: 'Books')
        categoria_de_la_fila = fila[col_agrupadora]
        
        # 3. Buscamos en nuestro mapa si tenemos la mediana para 'Books'
        # .get() busca la categoría; si no la encuentra, usa el 'respaldo_global'
        mediana_asignada = mapa_medianas.get(categoria_de_la_fila, respaldo_global)
        
        return mediana_asignada
    
    # PASO E: EJECUCIÓN (Pasar la regla fila por fila)
    # .apply(..., axis=1) significa: 'recorre el dataframe fila por fila horizontalmente'
    # Ejecutamos la regla tanto en el set de Entrenamiento como en el de Pruebas.
    X_train_out[col_a_imputar] = X_train_out.apply(
        lambda row: rellenar_vacio(row, medianas_por_categoria, Wood_respaldo := mediana_global_train), axis=1
    )
    X_test_out[col_a_imputar] = X_test_out.apply(
        lambda row: rellenar_vacio(row, medianas_por_categoria, mediana_global_train), axis=1
    )
    
    # PASO F: REPORTE PARA EL USUARIO
    # Imprime la auditoría final en pantalla para que el ingeniero verifique el éxito del proceso.
    print("="*60)
    print(f"🔬 REPORTE DE IMPUTACIÓN ANALÍTICA: '{col_a_imputar}'")
    print("="*60)
    print(f"📊 Nulos corregidos en X_train : {nulos_train_inicial:,} -> {X_train_out[col_a_imputar].isna().sum()}")
    print(f"📊 Nulos corregidos en X_test  : {nulos_test_inicial:,} -> {X_test_out[col_a_imputar].isna().sum()}")
    print("="*60)
    print("📌 Mapeo de medianas por categoría calculadas en Train:")
    for cat, val in medianas_por_categoria.items():
        print(f"   └─ Categoría: '{cat}' ➔ Mediana Aplicada: {val:.2f}")
    print("="*60)
    
    # Devolvemos los nuevos DataFrames ya reparados y listos
    return X_train_out, X_test_out

# Ejecución controlada usando tus variables exactas en minúsculas
X_train, X_test = ejecutar_imputacion_inteligente(
    X_train_df=X_train, 
    X_test_df=X_test, 
    col_agrupadora='category', 
    col_a_imputar='rating_avg'
)

🔬 REPORTE DE IMPUTACIÓN ANALÍTICA: 'rating_avg'
📊 Nulos corregidos en X_train : 0 -> 0
📊 Nulos corregidos en X_test  : 0 -> 0
📌 Mapeo de medianas por categoría calculadas en Train:
   └─ Categoría: 'Automotive' ➔ Mediana Aplicada: 4.70
   └─ Categoría: 'Beauty & Personal Care' ➔ Mediana Aplicada: 4.30
   └─ Categoría: 'Books' ➔ Mediana Aplicada: 3.90
   └─ Categoría: 'Clothing & Accessories' ➔ Mediana Aplicada: 3.90
   └─ Categoría: 'Electronics' ➔ Mediana Aplicada: 4.10
   └─ Categoría: 'Grocery & Gourmet' ➔ Mediana Aplicada: 4.00
   └─ Categoría: 'Home & Kitchen' ➔ Mediana Aplicada: 4.20
   └─ Categoría: 'Office Products' ➔ Mediana Aplicada: 4.10
   └─ Categoría: 'Sports & Outdoors' ➔ Mediana Aplicada: 4.00
   └─ Categoría: 'Toys & Games' ➔ Mediana Aplicada: 4.50


📊 ¿Por qué usamos la MEDIANA en vez de la MEDIA (Promedio)?
La razón es puramente estadística: la media es sumamente sensible a los valores extremos (outliers), mientras que la mediana es inmune a ellos.

Imaginemos que en la categoría de Electronics tenemos 5 productos con las siguientes calificaciones:
[1.0, 4.5, 4.7, 4.8, 5.0] (el 1.0 puede ser un producto defectuoso o una reseña malintencionada).

- Si calculamos la Media (Promedio): Sumamos todo y dividimos entre 5. El resultado es 4.00. Ese único 1.0 arrastró y bajó el promedio de toda la categoría, haciendo parecer que la electrónica es peor de lo que realmente es.

- Si calculamos la Mediana: Ordenamos los números de menor a mayor y tomamos el valor que se encuentra exactamente en el centro. El valor central es 4.70.

💡 Conclusión: La mediana representa mucho mejor al "producto típico" de una categoría porque ignora los extremos atípicos. Por eso, en analítica de datos, para rellenar nulos siempre se prefiere la mediana.

🛡️ El concepto clave: ¿Qué es el "Anti-Data Leakage" (Fuga de Datos)?
Si miramos el reporte, el script dice:

Nulos corregidos en X_train: 9,230 -> 0

Nulos corregidos en X_test: 2,385 -> 0

El script calculó las medianas de las categorías utilizando únicamente las 80,000 filas de X_train. El conjunto de examen (X_test) se quedó cerrado bajo llave. Una vez calculadas las medianas (por ejemplo, el 4.10 para Electronics), usamos ese dato para rellenar tanto los vacíos de Train como los de Test.

Si hubiésemos usado todo el dataset completo antes del split para calcular las medianas, la información del conjunto de prueba habría "contaminado" el entrenamiento. Eso es el Data Leakage. Al hacerlo como lo hicimos, garantizamos un modelo honesto que cuando sea evaluado con el set de Test, se enfrentará a datos verdaderamente desconocidos.

## 🎯 FASE 3: CONSTRUCCIÓN DEL PIPELINE DE TRANSFORMACIÓN AUTOMATIZADO

¿De qué se trata esta etapa y por qué la hacemos?
nuestro set de datos quedó dividido en dos grandes bloques técnicos que el modelo en su estado actual, no puede entender:

- Las 12 Variables Categóricas (object): Son texto puro (interaction_type, device_type, loyalty_tier, etc.). Los modelos no saben qué es "Smartphone" o "Premium". Necesitamos convertirlas en números (ceros y unos).

- Las 6 Variables Numéricas (int64/float64): Tienen escalas totalmente disparadas. price o income_level se mueven en miles, mientras que rating_avg se mueve entre 1 y 5, y age entre 18 y 80. Si las dejamos así, el modelo se confundirá y pensará que los ingresos son miles de veces más importantes que la calificación solo por el tamaño del número. Necesitamos estandarizarlas (nivelar el terreno de juego).

🛠️ Los dos motores del PipelinePara transformar los datos, montaremos dos motores específicos dentro de nuestra aduana:

- Para las variables numéricas ➔ StandardScaler (Estandarización): Restará la media y dividirá por la desviación estándar. Al hacer esto, todas las variables numéricas pasarán a moverse en un rango unificado (generalmente entre -3 y 3), donde el centro exacto será el 0.
- Para las variables categóricas ➔ OneHotEncoder (Codificación binaria): Creará columnas de presencia (sistema binario). Si device_type tiene "Mobile" y "Desktop", creará las columnas device_type_mobile y device_type_desktop. Además, añadiremos un seguro de vida profesional: handle_unknown='ignore'. Si en el set de prueba entra un país o una marca que el modelo jamás vio en el entrenamiento, el script no romperá, simplemente le asignará ceros.

🧑‍💻 Código de la Fase 3: El Transformador Maestro

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# =====================================================================
# FASE 3: PIPELINE DE TRANSFORMACIÓN AUTOMATIZADO (PREPARACIÓN FINAL)
# =====================================================================

print("="*60)
print("⚙️ CONFIGURANDO ADUANA DE PROCESAMIENTO (COLUMN TRANSFORMER)")
print("="*60)

# 1. Definición de los procesadores específicos para cada bloque técnico
PROCESADOR_NUMERICO = StandardScaler()
PROCESADOR_CATEGORICO = OneHotEncoder(drop=None, handle_unknown='ignore', sparse_output=False)

# 2. Construcción del ColumnTransformer (La aduana unificada)
# El formato es: ('nombre_clave', transformador_a_usar, lista_de_columnas_destino)
aduana_datos = ColumnTransformer(
    transformers=[
        ('num', PROCESADOR_NUMERICO, variables_numericas),
        ('cat', PROCESADOR_CATEGORICO, variables_categoricas)
    ],
    remainder='drop' # Seguridad: cualquier columna no declarada se descarta
)

# 3. EL AJUSTE SAGRADO (Ajustamos SOLO con Train para evitar Data Leakage)
print("🏋️‍♂️ Entrenando la aduana con los parámetros de X_train...")
aduana_datos.fit(X_train)

# 4. LA TRANSFORMACIÓN REPLICABLE
# Transformamos tanto Train como Test usando el conocimiento extraído de Train
print("🔄 Transformando matrices X_train y X_test al nuevo formato matemático...")
X_train_trans = aduana_datos.transform(X_train)
X_test_trans = aduana_datos.transform(X_test)

# 5. Extracción de los nuevos nombres de columnas (Para no perder la brújula)
nuevos_nombres_columnas = aduana_datos.get_feature_names_out()

# 6. REPORTE EJECUTIVO DE LA FASE 3
print("="*60)
print("🔬 REPORTE DE TRANSFORMACIÓN E INGENIERÍA DE COMPONENTES")
print("="*60)
print(f"📊 Dataset Original (X_train) : {X_train.shape[0]:,} filas × {X_train.shape[1]} columnas")
print(f"🚀 Dataset Procesado (X_train_trans): {X_train_trans.shape[0]:,} filas × {X_train_trans.shape[1]} columnas")
print("-" * 60)
print(f"💡 ¡La dimensionalidad se expandió de {X_train.shape[1]} a {X_train_trans.shape[1]} columnas!")
print("   (Esto se debe a la conversión de textos a columnas binarias 0/1)")
print("="*60)

⚙️ CONFIGURANDO ADUANA DE PROCESAMIENTO (COLUMN TRANSFORMER)
🏋️‍♂️ Entrenando la aduana con los parámetros de X_train...
🔄 Transformando matrices X_train y X_test al nuevo formato matemático...
🔬 REPORTE DE TRANSFORMACIÓN E INGENIERÍA DE COMPONENTES
📊 Dataset Original (X_train) : 80,000 filas × 18 columnas
🚀 Dataset Procesado (X_train_trans): 80,000 filas × 5893 columnas
------------------------------------------------------------
💡 ¡La dimensionalidad se expandió de 18 a 5893 columnas!
   (Esto se debe a la conversión de textos a columnas binarias 0/1)


🔍 1. ¿Por qué se ha expandido tanto el dataset?
El culpable absoluto de esta explosión de columnas es el OneHotEncoder actuando sobre variables con una alta cardinalidad (columnas que tienen demasiados valores únicos distintos).

Si revisamos la lista de variables categóricas que detectamos en la Fase 1:

Columnas como gender (2 valores) o device_type (3 o 4 valores) solo aportan un puñado de columnas nuevas.

Pero columnas como city (ciudades), brand (marcas) y subcategory (subcategorías) probablemente tienen cientos o miles de valores distintos en un dataset de 100,000 registros. El codificador ha creado una columna independiente de ceros y unos para cada ciudad y cada marca del mundo que aparecía en los datos.

⚠️ 2. ¿Por qué es un problema grave tener 5,893 columnas?
Si intentamos alimentar a la mayoría de los modelos de Machine Learning (como una Regresión Logística, un Árbol de Decisión o KNN) con casi 6,000 variables para solo 80,000 filas, nos enfrentaremos a tres problemas críticos:

Overfitting (Sobreajuste): El modelo se volverá "perezoso". En lugar de aprender patrones generales de comportamiento de compra, se memorizará combinaciones absurdas y ultra-específicas (ej: "si el cliente es de la ciudad X, compra la marca Y, el martes a las 5, entonces compra"). Funcionará perfecto en Train, pero fracasará rotundamente en Test.

Dispersión de Datos (Sparsity): Tus matrices ahora están inundadas de millones de ceros (0) y muy pocos unos (1). Matemáticamente, esto diluye la señal predictiva de variables poderosas como el precio o el tiempo de retención.

Coste Computacional: El entrenamiento se volverá sumamente lento y consumirá toda la memoria RAM de tu equipo de forma innecesaria.

🛠️ ¿Cómo lo solucionamos de forma profesional?
No podemos ir hacia atrás; el pipeline automatizado es correcto, pero necesita estrategia de negocio. Para reducir esas 5,893 columnas a un número saludable (menos de 50 o 100), tenemos dos caminos metodológicos:

Opción A (Filtrado de Cardinalidad en el EDA): Agrupar los valores minoritarios de columnas como city o brand. Por ejemplo, mantener las 10 ciudades con más ventas y todas las demás transformarlas automáticamente en una categoría llamada 'Other'.

Opción B (Eliminación Defensiva): Evaluar si variables como city o brand realmente aportan valor predictivo de cara a la conversión, o si eliminándolas del set de predictores (columnas_a_descartar en la Fase 1) limpiamos el ruido de golpe manteniendo la esencia del comportamiento del usuario (interaction_type, loyalty_tier, price, dwell_time_secs).

## 🛠️ El Próximo Paso Técnico: Refactorización Defensiva del Pipeline
Aplicando los descubrimientos de nuestro informe ejecutivo (donde vimos que el peso real está en el País, el Rango de Edad, la Categoría y el Comportamiento), la explosión de columnas se debe a variables de altísima cardinalidad que meten ruido estático: probablemente brand (marcas específicas) o city (ciudades individuales).

¿Qué compra un usuario de 70 años en EE. UU.? Eso nos importa. ¿Si vive en la micro-ciudad X o Y? Eso diluye el modelo.

Nuestra solución técnica será una estrategia mixta:

Eliminación Estratégica: Vamos a sacar del entrenamiento variables ultra-específicas que no aportan al comportamiento genérico (como identificadores o ciudades/marcas si estas dispersan la matriz).

Control de Frecuencia (Agrupación en 'Other'): Configurar nuestro OneHotEncoder o mapear el dataset antes de la aduana para mantener intactas las categorías clave (como country, category, rango_edad) y agrupar los elementos minoritarios.

💻 El Código de Intervención Quirúrgica

In [22]:
# Inspección rápida para verificar los nombres reales de las columnas
print("📋 Columnas actuales en X_train:")
print(list(X_train.columns))

📋 Columnas actuales en X_train:
['interaction_type', 'device_type', 'referrer_source', 'age', 'gender', 'country', 'city', 'income_level', 'preferred_category', 'loyalty_tier', 'category', 'subcategory', 'brand', 'price', 'rating_avg', 'review_count', 'stock_quantity', 'dwell_time_secs']


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# =====================================================================
# INTERVENCIÓN EJECUTIVA: SANEAMIENTO DE CARDINALIDAD (FASE 3 - CORREGIDA)
# =====================================================================

print("="*60)
print("🛡️ APLICANDO FILTRADO EJECUTIVO DE VARIABLES (REDUCCIÓN DE RUIDO)")
print("="*60)

# 1. AJUSTE DE BRÚJULA: Usamos solo las variables reales y existentes en tu X_train
# Agregamos rating_avg y review_count que aportan gran valor predictivo
variables_numericas = ['price', 'dwell_time_secs', 'age', 'rating_avg', 'review_count'] 

# Mantenemos las columnas clave de negocio, descartando 'city' y 'brand' para evitar las 5,893 columnas
variables_categoricas = ['country', 'category', 'device_type', 'interaction_type', 'loyalty_tier']

# 2. Configuración del OneHotEncoder con el parámetro de seguridad max_categories
# 'min_frequency=0.01' le dice al codificador: "Si un país o categoría representa menos del 1% del negocio, 
# agrúpalo automáticamente en una columna llamada 'infrequent_subset'". ¡Adiós a las miles de columnas muertas!
PROCESADOR_NUMERICO = StandardScaler()
PROCESADOR_CATEGORICO = OneHotEncoder(
    drop='first',                  # Eliminamos la primera columna de cada categoría para evitar redundancia
    handle_unknown='infrequent_if_exist', 
    min_frequency=0.01,            # Agrupa categorías minoritarias (<1%) en una sola columna residual
    sparse_output=False
)

# 3. Reconstrucción de la Aduana Unificada (ColumnTransformer)
aduana_datos = ColumnTransformer(
    transformers=[
        ('num', PROCESADOR_NUMERICO, variables_numericas),
        ('cat', PROCESADOR_CATEGORICO, variables_categoricas)
    ],
    remainder='drop' # Todo lo que no esté en nuestras variables estratégicas se descarta por seguridad
)

# 4. EL AJUSTE SAGRADO SOBRE X_TRAIN (Sin Data Leakage)
print("🏋️‍♂️ Entrenando la nueva aduana optimizada...")
aduana_datos.fit(X_train)

# 5. TRANSFORMACIÓN EFICIENTE
print("🔄 Transformando matrices X_train y X_test al nuevo formato matemático...")
X_train_trans = aduana_datos.transform(X_train)
X_test_trans = aduana_datos.transform(X_test)

# Extracción de nombres de las nuevas columnas para el modelo predictivo
nuevos_nombres_columnas = aduana_datos.get_feature_names_out()

# 6. REPORTE DE SALIDA SANEADO
print("="*60)
print("🔬 REPORTE DE TRANSFORMACIÓN OPTIMIZADO PARA MACHINE LEARNING")
print("="*60)
print(f"📊 Dataset Original (X_train) : {X_train.shape[0]:,} filas × {X_train.shape[1]} columnas")
print(f"🚀 Dataset Procesado Saneado : {X_train_trans.shape[0]:,} filas × {X_train_trans.shape[1]} columnas")
print("-" * 60)
print(f"💡 ¡Hemos consolidado la potencia predictiva en {X_train_trans.shape[1]} columnas limpias!")
print("   Listos para entrenar algoritmos de clasificación estables sin Overfitting.")
print("="*60)

🛡️ APLICANDO FILTRADO EJECUTIVO DE VARIABLES (REDUCCIÓN DE RUIDO)
🏋️‍♂️ Entrenando la nueva aduana optimizada...
🔄 Transformando matrices X_train y X_test al nuevo formato matemático...
🔬 REPORTE DE TRANSFORMACIÓN OPTIMIZADO PARA MACHINE LEARNING
📊 Dataset Original (X_train) : 80,000 filas × 18 columnas
🚀 Dataset Procesado Saneado : 80,000 filas × 43 columnas
------------------------------------------------------------
💡 ¡Hemos consolidado la potencia predictiva en 43 columnas limpias!
   Listos para entrenar algoritmos de clasificación estables sin Overfitting.


Hola Gemini, soy Carlos. Vamos a continuar con nuestro proyecto de Data Analytics y Machine Learning para E-commerce. Activa tu rol de Director Ejecutivo de Análisis de Datos y toma nota de nuestro estado actual para guiarme en el siguiente paso:

1. ¿Dónde estamos?: Hemos terminado por completo la Fase 3 (Procesamiento de Datos) en el cuaderno `procesamiento.ipynb`.
2. El hito técnico: Solucionamos una explosión de dimensionalidad provocada por el OneHotEncoder, logrando reducir con éxito el dataset de 5,893 columnas a 43 columnas limpias y optimizadas, aplicando filtros de frecuencia y manteniendo las variables estratégicas (país, categoría, rango_edad, etc.). El script corrió perfectamente sin errores.
3. El enfoque de negocio: Ya presentamos el informe ejecutivo. Nuestras dos prioridades estratégicas validadas son:
   - Optimizar la pasarela de pago en EE. UU. (añadiendo One-Click checkout como Apple Pay/PayPal) para corregir su conversión del 10.38%.
   - Inyectar volumen de tráfico al "Sweet Spot" demográfico senior (63-75 años) que tiene la conversión más alta del negocio (13.86%).
4. Siguiente paso inmediato: Arrancar la Fase 4 (Modelado Predictivo e Inteligencia Artificial). Me quedé justo por ejecutar la celda para entrenar el modelo base de Regresión Logística utilizando las matrices transformadas con las 43 columnas.

Por favor, confirma que tienes todo el contexto claro, hazme un breve resumen de lo que haremos hoy en la Fase 4 y dime cómo arrancamos.